In [3]:
import torch
x = torch.zeros(4,8)

In [4]:
x.numel()

32

In [5]:
a = torch.tensor([1e-8], dtype = torch.float16)
assert a == 0

In [6]:
a = torch.tensor([1e-8], dtype = torch.bfloat16)
assert a != 0

In [7]:
torch.finfo(torch.float32)

finfo(resolution=1e-06, min=-3.40282e+38, max=3.40282e+38, eps=1.19209e-07, smallest_normal=1.17549e-38, tiny=1.17549e-38, dtype=float32)

In [8]:
torch.finfo(torch.float16)

finfo(resolution=0.001, min=-65504, max=65504, eps=0.000976562, smallest_normal=6.10352e-05, tiny=6.10352e-05, dtype=float16)

In [9]:
torch.finfo(torch.bfloat16)

finfo(resolution=0.01, min=-3.38953e+38, max=3.38953e+38, eps=0.0078125, smallest_normal=1.17549e-38, tiny=1.17549e-38, dtype=bfloat16)

In [10]:
from einops import *

x = torch.ones(3,4)
y = torch.ones(4,3)

# Old way
z = x @ y

In [11]:
# New way

z_einops = einsum(x, y, "seq1 hidden, hidden seq2 -> seq1 seq2")

In [12]:
z == z_einops

tensor([[True, True, True],
        [True, True, True],
        [True, True, True]])

In [13]:
x = torch.ones(2,3,4)
y = torch.ones(2,3,4)

In [14]:
z_old = x @ y.transpose(-2,-1)

In [15]:
z_ei = einsum(x,y, "batch seq1 hidden, batch seq2 hidden -> batch seq1 seq2" )

In [16]:
x = torch.ones(3,8)
x = rearrange(x, "... (heads hidden1) -> ... heads hidden1", heads = 2)
x.shape

torch.Size([3, 2, 4])

In [17]:
B = 1024
D = 256
K = 64
device = "cuda"
x = torch.ones(B,D, device=device)
w = torch.ones(D,K, device= device)

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
y = x @ w

How Many flops? 
A single scalar product takes D mul and D-1 sum, suppose D 2 * D, then I have to do it for every columns and for every row, so total flops = 2 * D * B* K

In [ ]:
import timeit

B = 4096
D = 4096
K = 4096
actual_num_flops = 2 * D * B * K
x = torch.ones(B,D, device=device,dtype=torch.bfloat16)
w = torch.ones(D,K, device= device,dtype=torch.bfloat16)
def run():
   x @ w 
   torch.cuda.synchronize()
num_trials = int(10)
total_time = timeit.timeit(run, number = num_trials)
actual_time = total_time/num_trials
actual_flop_per_sec = actual_num_flops / actual_time
actual_flop_per_sec

3960378379571.6504

In [ ]:
actual_num_flops = 2 * D * B * K

In [ ]:
actual_flop_per_sec = actual_num_flops / actual_time

In [ ]:
actual_flop_per_sec

273797416893.00964

In [25]:
# Params accounting 
def params_account(V, N, D, F):

   P = N * (4 * D**2 + 2 * D + 3* D *F) + 2* V* D + D
   return P



In [ ]:
#GPT2 xl
params_gpt2_xl = params_account(50257, 48, 1600, 4288)
params_gpt2_xl


1640452800

In [ ]:
x = params_gpt2_xl
print(f"{x:e}")

1.640453e+09


In [ ]:
# in fp32 4 bytes for each parameter
bytes = 4 * x
GBib = (bytes / ( 2 ** 30))
print(f"{GBib:e}")



6.111163e+00


In [19]:
def return_flops(D, T, V, N, F):
   block = 8 * T * D**2 + 4 * T**2 * D + 6 * T * D * F
   full  = N * block + 2 * T * D * V
   return full

In [20]:
FLOPs_gpt2_xl = return_flops(1600, 1024, 50257, 48, 4288)
print(f"{FLOPs_gpt2_xl:e}")

3.516770e+12


In [21]:
def decompose_flops(D, T, V, N, F):
    FFN_cost = 6 * T * D * F * N
    Final_projection_layer = 2 * T * D * V
    mha = N * (8 * T * D**2 + 4 * T**2 * D)

    total_cost = return_flops(D, T, V, N, F)

    components = {
        "FFN": FFN_cost,
        "Final projection layer": Final_projection_layer,
        "MHA": mha,
    }

    print(f"Total FLOPs: {total_cost:,.0f}")
    print("-" * 65)

    for name, cost in components.items():
        percentage = 100 * cost / total_cost
        print(f"{name:<30} {cost:>22,.0f}  ({percentage:6.2f}%)")

    print("-" * 65)

    accounted_for = sum(components.values())

    print(
        f"{'Total accounted for':<30} "
        f"{accounted_for:>22,.0f}  "
        f"({100 * accounted_for / total_cost:6.2f}%)"
    )

    return components, total_cost



In [22]:
models = {
    "GPT-2 Small": {
        "D": 768,
        "T": 1024,
        "V": 50_257,
        "N": 12,
        "F": round((8 / 3 * 768) / 64) * 64,
    },
    "GPT-2 Medium": {
        "D": 1024,
        "T": 1024,
        "V": 50_257,
        "N": 24,
        "F": round((8 / 3 * 1024) / 64) * 64,
    },
    "GPT-2 Large": {
        "D": 1280,
        "T": 1024,
        "V": 50_257,
        "N": 36,
        "F": round((8 / 3 * 1280) / 64) * 64,
    },
    "GPT-2 XL": {
        "D": 1600,
        "T": 1024,
        "V": 50_257,
        "N": 48,
        "F": 4288,
    },
}


for model_name, cfg in models.items():
   print("\n" + "=" * 70)
   print(model_name)
   print("=" * 70)

   print(
        f"D={cfg['D']}, "
        f"T={cfg['T']}, "
        f"V={cfg['V']:,}, "
        f"N={cfg['N']}, "
        f"F={cfg['F']}"
   )

   decompose_flops(
        D=cfg["D"],
        T=cfg["T"],
        V=cfg["V"],
        N=cfg["N"],
        F=cfg["F"],
   )


GPT-2 Small
D=768, T=1024, V=50,257, N=12, F=2048.0
Total FLOPs: 291,648,307,200
-----------------------------------------------------------------
FFN                                   115,964,116,992  ( 39.76%)
Final projection layer                 79,047,426,048  ( 27.10%)
MHA                                    96,636,764,160  ( 33.13%)
-----------------------------------------------------------------


TypeError: unsupported format string passed to dict_values.__format__

In [23]:
# --------------------------------------------------
# GPT-2 XL configuration
# --------------------------------------------------

D = 1600
V = 50_257
N = 48
F = 4288

context_lengths = [1024, 16_384]

results = {}

for T in context_lengths:
   print("\n" + "=" * 70)
   print(f"GPT-2 XL, context length T = {T:,}")
   print("=" * 70)

   components, total = decompose_flops(
        D=D,
        T=T,
        V=V,
        N=N,
        F=F,
   )

   results[T] = {
        "components": components,
        "total": total,
   }


# --------------------------------------------------
# Compare T = 1024 with T = 16384
# --------------------------------------------------

old_total = results[1024]["total"]
new_total = results[16_384]["total"]

factor = new_total / old_total
percent_increase = 100 * (new_total - old_total) / old_total

print("\n" + "=" * 70)
print("CHANGE IN TOTAL FLOPs")
print("=" * 70)

print(f"T = 1,024:  {old_total:,.0f} FLOPs")
print(f"T = 16,384: {new_total:,.0f} FLOPs")

print(f"\nIncrease factor: {factor:.2f}x")
print(f"Percentage increase: {percent_increase:.2f}%")


GPT-2 XL, context length T = 1,024
Total FLOPs: 3,516,769,894,400
-----------------------------------------------------------------
FFN                                 2,023,332,249,600  ( 57.53%)
Final projection layer                164,682,137,600  (  4.68%)
MHA                                 1,328,755,507,200  ( 37.78%)
-----------------------------------------------------------------


TypeError: unsupported format string passed to dict_values.__format__

In [ ]:
#Resource accounting for adamw

#batch_size and the model hyperparameters (vocab_size,
#context_length, num_layers, d_model, num_heads

# As now i have already the function that computes the numbner of parameters. Only for them
# we need P + P(grad) + P(m) + P (v) = 4P -> considering 4 bytes for each one 
# 16P bytes only for params and gradient, now I have to account for activations

In [27]:
def return_adamw_memory_usage(batch_size, vocab_size, context_length, num_layers, d_model, num_heads):
   B = batch_size
   T = context_length
   D = d_model
   V = vocab_size
   N = num_layers
   H = num_heads
   F = (8 / 3) * D

   params = params_account(V, N, D, F)
    # Parameters, gradients, Adam m, Adam v
   params_memory = 4 * params

    # Activation storage for one Transformer block
   one_block = 8 * B * T * D + 2 * B * H * T**2 + 4 * B * T * F

    # Total activation storage
   activation_memory = N * one_block + B * T * D + B * T * V + B * T

   return (params_memory + activation_memory) * 4

In [30]:
V=50257
T=1024
N=48
D=1600
H=25
B = 1
while return_adamw_memory_usage(B, V, T, N, D, H) < 80 * 10**9:
   B+=1

B-1  


3

In [29]:
print(N, type(N))
print(D, type(D))
print(F, type(F))

(48,) <class 'tuple'>
1600 <class 'int'>
4288 <class 'int'>


In [ ]:
#How many FLOPs does running one step of AdamW take?
# I can account for one step for a single parameter and multiply by the number of parameters
# roughly 25 flops for each param

In [33]:
# An NVIDIA H100 GPU has a theoretical peak of 495 teraFLOP/
# s for “float32” (actually TensorFloat-32, which in reality is “bfloat19”) operations. Assuming
# you are able to get 50% MFU, how long would it take to train a GPT-2 XL for 400K steps
# and a batch size of 1024 on a single H100? Following J. Kaplan et al. [25] and
# J. Hoffmann et al. [26], assume that the backward pass has twice the FLOPs of the forward
# pass

FLOPS = 495 * 10**12 * 0.5
steps = 400 * 10**3
B = 1024
V = 50257
T = 1024
N = 48
D = 1600
H = 25
F = 4288
# I can account for the flops needed for a single optimization steps, in which I have
# Flops Forward + Flops Backward(2 * Fowrard) + Flops Optimization(Linear in the number of parameters)

Flops_forward = B * return_flops(D, T, V, N, F)
num_params = params_account(V, N, D, F)
total_flops = (3 * Flops_forward + 15 * num_params) * steps


hour_training_time = ((total_flops) / FLOPS ) / 3600


In [34]:
hour_training_time

4850.074847312592